# BMI5109 Group 4 – Sepsis Risk Prediction from ICU Time-Series Data
**End-to-End Pipeline** · NumPy / Scikit-learn · All outputs pre-computed and embedded

| Step | Description |
|------|-------------|
| 1 | Data Loading (folder, zip, or GUI picker) |
| 2 | Exploratory Data Analysis |
| 3 | Pre-processing & Missingness Imputation |
| 4 | Feature Engineering |
| 5 | Statistical Analysis → **p-value table** |
| 6 | Train / Test Split |
| 7 | Logistic Regression & Random Forest |
| 8 | NumPy GRU & LSTM |
| 9 | Model Evaluation (AUROC, AUPRC, Sens, Spec) |
| 10 | Clinical Early Warning Metrics |
| 11 | Feature Importance |
| 12 | Publication Figures |

## Cell 1 – Imports

In [ ]:
import os, sys, zipfile, glob, warnings, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from IPython.display import display
from collections import defaultdict
from scipy.stats import mannwhitneyu, chi2_contingency, fisher_exact
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    roc_curve, precision_recall_curve, confusion_matrix
)
warnings.filterwarnings("ignore")
np.random.seed(42)
%matplotlib inline
print("All imports OK")

## Cell 2 – Configuration 

Set `DATA_SOURCE` to one of:
- **Option A** – path to folder of `.psv` files: `r"C:\\Users\\YourName\\Desktop\\Data"`
- **Option B** – path to `.zip` file: `r"C:\\Users\\YourName\\Desktop\\Data.zip"`
- **Option C** – `None` → GUI folder/file picker opens automatically

Set `OUTPUT_DIR` to where you want results saved.

In [ ]:
import os

# ── BASE DIRECTORY (Jupyter working dir) ───────────────────────────────
BASE_DIR = os.getcwd()

# ── PATH CONFIG ────────────────────────────────────────────────────────
DATA_SOURCE = os.path.join(BASE_DIR, "Dataset.csv")   # put CSV in same folder
OUTPUT_DIR  = os.path.join(BASE_DIR, "sepsis_results")

# Create output folder if not exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"[INFO] Base dir: {BASE_DIR}")
print(f"[INFO] Data path: {DATA_SOURCE}")
print(f"[INFO] Output dir: {OUTPUT_DIR}")

# ── DATA SETTINGS ─────────────────────────────────────────────────────
PATIENT_ID_COL = "patient_id"
SEPARATOR      = ","

VITAL_COLS = ["HR","O2Sat","Temp","SBP","MAP","DBP","Resp"]
LAB_COLS   = [
    "EtCO2","BaseExcess","HCO3","FiO2","pH","PaCO2","SaO2",
    "AST","BUN","Alkalinephos","Calcium","Chloride","Creatinine",
    "Bilirubin_direct","Glucose","Lactate","Magnesium","Phosphate",
    "Potassium","Bilirubin_total","TroponinI","Hct","Hgb","PTT",
    "WBC","Fibrinogen","Platelets"
]

ALL_FEATURES = VITAL_COLS + LAB_COLS
LABEL_COL    = "SepsisLabel"

# ── MODEL CONFIG ──────────────────────────────────────────────────────
SEQ_LEN    = 8
EPOCHS_RNN = 5
LR_RNN     = 0.005
HIDDEN_DIM = 32

In [ ]:
import os
print(os.getcwd())

## Step 1 – Data Loading

In [ ]:
DATA_SOURCE = "Dataset.csv"

def _load_flat_csv(source):
    source = os.path.expanduser(str(source).strip())
    if not os.path.isfile(source):
        raise FileNotFoundError(f"File not found: {source}")

    raw = pd.read_csv(source, sep=SEPARATOR, engine="python")

    if PATIENT_ID_COL and PATIENT_ID_COL in raw.columns:
        raw = raw.rename(columns={PATIENT_ID_COL: "patient_id"})
    elif "patient_id" not in raw.columns:
        if "ICULOS" in raw.columns:
            reset = raw["ICULOS"].diff().fillna(0) < 0
            raw["patient_id"] = (reset | (raw["ICULOS"] == 1)).cumsum().astype(str).apply(lambda x: f"p{int(x):05d}")
        else:
            raw["patient_id"] = "p00001"

    if "ICULOS" not in raw.columns:
        raw["ICULOS"] = raw.groupby("patient_id").cumcount() + 1

    if LABEL_COL not in raw.columns:
        raw[LABEL_COL] = 0

    return raw

raw_df = _load_flat_csv(DATA_SOURCE)
print(raw_df.shape)
print(raw_df["patient_id"].nunique())

## Step 2 3 – Exploratory Data Analysis

In [ ]:

eda_df = raw_df.copy().sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

# 1) Basic summaries
pat_level = eda_df.groupby("patient_id")[LABEL_COL].max()
n_patients = len(pat_level)
n_sep = int((pat_level == 1).sum())
stay_len = eda_df.groupby("patient_id")["ICULOS"].max()
miss_pct = eda_df[ALL_FEATURES].isnull().mean().sort_values(ascending=False) * 100

print(f"Sepsis patients       : {n_sep} / {n_patients} ({100*n_sep/n_patients:.1f}%)")
print(f"ICU stay median       : {stay_len.median():.0f}h (range {stay_len.min()}–{stay_len.max()}h)")
print(f"Positive label rate   : {eda_df[LABEL_COL].mean()*100:.2f}%")
print("\nTop 10 missing features:")
print(miss_pct.head(10).round(2))

# 2) Missingness plot
fig, ax = plt.subplots(figsize=(14, 5))
colors = plt.cm.Reds(miss_pct.values / 100)
miss_pct.plot.bar(ax=ax, color=colors, edgecolor="white")
ax.set_ylabel("Missing rate (%)")
ax.set_title("Feature Missingness Profile")
ax.set_ylim(0, 105)
ax.axhline(50, color="red", lw=1, ls="--", label="50% threshold")
ax.legend()
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

# 3) Patient-level snapshot table for boxplots
sepsis_first = (
    eda_df.loc[eda_df[LABEL_COL] == 1]
    .sort_values(["patient_id", "ICULOS"])
    .groupby("patient_id", as_index=False)
    .first()
)
sepsis_first["group"] = "Sepsis"

nonsepsis_last = (
    eda_df.groupby("patient_id")[LABEL_COL].max()
    .loc[lambda s: s == 0]
    .index
)
nonsepsis_last = (
    eda_df.loc[eda_df["patient_id"].isin(nonsepsis_last)]
    .sort_values(["patient_id", "ICULOS"])
    .groupby("patient_id", as_index=False)
    .last()
)
nonsepsis_last["group"] = "No Sepsis"

snapshot_df = pd.concat([sepsis_first, nonsepsis_last], ignore_index=True)

# 4) Boxplots
plot_cols = ["HR", "MAP", "Lactate", "Resp"]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, plot_cols):
    g0 = snapshot_df.loc[snapshot_df["group"] == "No Sepsis", col].dropna()
    g1 = snapshot_df.loc[snapshot_df["group"] == "Sepsis", col].dropna()

    ax.boxplot(
        [g0, g1],
        labels=["No Sepsis", "Sepsis"],
        patch_artist=True,
        boxprops=dict(facecolor="#cce5ff"),
        medianprops=dict(color="red")
    )
    ax.set_title(col)
    ax.set_ylabel(col)

plt.suptitle("Patient-Level Snapshot Comparison", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Step 3 - Preprocessing / Imputation

df = raw_df.copy().sort_values(["patient_id", "ICULOS"])

# Save original missingness before filling
orig_missing = df[ALL_FEATURES].isnull().copy()

# Forward fill within each patient
df[ALL_FEATURES] = df.groupby("patient_id")[ALL_FEATURES].ffill()

# Median fill for remaining missing values
medians = df[ALL_FEATURES].median()
df[ALL_FEATURES] = df[ALL_FEATURES].fillna(medians)

# Missingness indicators
for col in ALL_FEATURES:
    df[f"miss_{col}"] = orig_missing[col].astype(int)

MISS_COLS = [f"miss_{c}" for c in ALL_FEATURES]

print(f"Remaining NaN after imputation: {df[ALL_FEATURES].isnull().sum().sum()}")
print(f"Missingness indicators added  : {len(MISS_COLS)}")

display(df[ALL_FEATURES].describe().round(2))

## Step 4 – Feature Engineering

In [ ]:
# =========================
# Step 4 – Fair Feature Engineering for LR/RF
# FAST VERSION (vectorized rolling-window features)
# =========================

import numpy as np
import pandas as pd
import time

# -------------------------
# 0) Debug / subset setting
# -------------------------
USE_SUBSET = True
DEBUG_N_PATIENTS = 3000
WINDOW_LEN = 12

# -------------------------
# 1) Choose working dataframe
# -------------------------
if USE_SUBSET:
    sample_pids = (
        df["patient_id"]
        .drop_duplicates()
        .sample(n=min(DEBUG_N_PATIENTS, df["patient_id"].nunique()), random_state=42)
    )
    df_work = df[df["patient_id"].isin(sample_pids)].copy()
    print("Using SUBSET mode")
else:
    df_work = df.copy()
    print("Using FULL dataset mode")

df_work = df_work.sort_values(["patient_id", "ICULOS"]).reset_index(drop=True)

print("Original df shape:", df.shape)
print("Working df shape :", df_work.shape)
print("Working patients :", df_work["patient_id"].nunique())

start_time = time.time()

# -------------------------
# 2) Row-level derived features first
# -------------------------
df_work["ShockIndex"] = df_work["HR"] / df_work["SBP"].replace(0, np.nan)
df_work["ShockIndex"] = df_work["ShockIndex"].fillna(df_work["ShockIndex"].median())

df_work["high_resp"] = (df_work["Resp"] > 22).astype(int)
df_work["low_MAP"] = (df_work["MAP"] < 70).astype(int)
df_work["high_lactate"] = (df_work["Lactate"] > 2).astype(int)
df_work["high_creat"] = (df_work["Creatinine"] > 1.2).astype(int)

BASE_TABULAR_FEATS = ALL_FEATURES + [
    "ShockIndex",
    "high_resp",
    "low_MAP",
    "high_lactate",
    "high_creat"
]

print(f"Base row-level features used to build windows: {len(BASE_TABULAR_FEATS)}")

# -------------------------
# 3) Vectorized 12h window feature construction
# Each row becomes the ENDPOINT of a 12h window
# -------------------------
g = df_work.groupby("patient_id", group_keys=False)

# position within each patient trajectory
df_work["row_in_patient"] = g.cumcount()

for idx, col in enumerate(BASE_TABULAR_FEATS, start=1):
    s = g[col]

    # last value at current hour
    df_work[f"{col}_last"] = df_work[col]

    # rolling stats over previous 12 rows within patient
    df_work[f"{col}_mean12"] = s.transform(
        lambda x: x.rolling(window=WINDOW_LEN, min_periods=WINDOW_LEN).mean()
    )
    df_work[f"{col}_std12"] = s.transform(
        lambda x: x.rolling(window=WINDOW_LEN, min_periods=WINDOW_LEN).std(ddof=0)
    )
    df_work[f"{col}_min12"] = s.transform(
        lambda x: x.rolling(window=WINDOW_LEN, min_periods=WINDOW_LEN).min()
    )
    df_work[f"{col}_max12"] = s.transform(
        lambda x: x.rolling(window=WINDOW_LEN, min_periods=WINDOW_LEN).max()
    )

    # first value in the 12h window = current value shifted by 11 within patient
    first_in_window = s.shift(WINDOW_LEN - 1)
    df_work[f"{col}_delta12"] = df_work[col] - first_in_window

    if idx % 10 == 0 or idx == len(BASE_TABULAR_FEATS):
        elapsed = time.time() - start_time
        print(f"[{idx}/{len(BASE_TABULAR_FEATS)}] features processed | elapsed: {elapsed/60:.2f} min")

# -------------------------
# 4) Keep only valid 12h window endpoints
# -------------------------
tab_df = df_work[df_work["row_in_patient"] >= WINDOW_LEN - 1].copy()

# columns to keep
TABULAR_WINDOW_FEATS = []
for col in BASE_TABULAR_FEATS:
    TABULAR_WINDOW_FEATS.extend([
        f"{col}_last",
        f"{col}_mean12",
        f"{col}_std12",
        f"{col}_min12",
        f"{col}_max12",
        f"{col}_delta12"
    ])

tab_df = tab_df[["patient_id", "ICULOS", LABEL_COL] + TABULAR_WINDOW_FEATS].copy()
tab_df = tab_df.rename(columns={"ICULOS": "ICULOS_end"})

elapsed_total = time.time() - start_time
print("\nFinished building tab_df.")
print("Window-level tabular dataset shape:", tab_df.shape)
print("Unique patients in tab_df:", tab_df["patient_id"].nunique())
print("Positive rate in tab_df (%):", round(tab_df[LABEL_COL].mean() * 100, 4))
print(f"Total LR/RF window-level features: {len(TABULAR_WINDOW_FEATS)}")
print(f"Total elapsed time: {elapsed_total/60:.2f} min")
print("Example feature names:", TABULAR_WINDOW_FEATS[:10])

## Step 5 – Statistical Analysis: p-value Table

In [ ]:
def get_diag_row(grp):
    s = grp[grp[LABEL_COL]==1]
    return s.iloc[0] if len(s)>0 else grp.iloc[-1]

diag_df = df.groupby("patient_id").apply(get_diag_row).reset_index(drop=True)
diag_df["sepsis"] = diag_df[LABEL_COL]
sep_g   = diag_df[diag_df["sepsis"]==1]
nsep_g  = diag_df[diag_df["sepsis"]==0]
print(f"Sepsis snapshots: {len(sep_g)}  |  No-sepsis snapshots: {len(nsep_g)}")

STAT_FEATS = ALL_FEATURES + ["Age","Gender","ShockIndex",
                              "high_resp","low_MAP","high_lactate","high_creat"]
rows = []
for feat in STAT_FEATS:
    if feat not in diag_df.columns: continue
    sv  = sep_g[feat].dropna(); nv = nsep_g[feat].dropna()
    if len(sv)<5 or len(nv)<5: continue
    uniq = diag_df[feat].dropna().nunique()
    if uniq <= 2:
        ct = pd.crosstab(diag_df["sepsis"], diag_df[feat])
        if ct.shape==(2,2): _, p = fisher_exact(ct.values); test="Fisher"
        else: _, p, _, _ = chi2_contingency(ct.values); test="Chi-sq"
        s_str = f"{sv.mean():.3f}"; ns_str = f"{nv.mean():.3f}"
    else:
        _, p = mannwhitneyu(sv, nv, alternative="two-sided"); test="MWU"
        s_str  = f"{sv.median():.3f} [{sv.quantile(.25):.3f}–{sv.quantile(.75):.3f}]"
        ns_str = f"{nv.median():.3f} [{nv.quantile(.25):.3f}–{nv.quantile(.75):.3f}]"
    sig = "***" if p<0.001 else ("**" if p<0.01 else ("*" if p<0.05 else "ns"))
    rows.append(dict(Feature=feat,Sepsis=s_str,NoSepsis=ns_str,Test=test,p_value=p,Sig=sig))

stat_df = pd.DataFrame(rows).sort_values("p_value")
stat_df.to_csv(os.path.join(OUTPUT_DIR,"pvalue_table.csv"), index=False)
print(f"Saved → {os.path.join(OUTPUT_DIR, 'pvalue_table.csv')}")

def style_sig_row(row):
    c = {"***":"#d4edda","**":"#fff3cd","*":"#ffeeba","ns":"#f8f9fa"}.get(row["Sig"],"#f8f9fa")
    return [f"background-color:{c}"]*len(row)

display(stat_df.style.apply(style_sig_row, axis=1)
               .format({"p_value":"{:.4e}"})
               .set_caption("Table 1: Covariable comparison at hour of sepsis diagnosis")
               .hide(axis="index"))

## Step 6 – Train / Test Split 

In [ ]:
# =========================
# Step 6 – Train / Test Split (fair version)
# =========================

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# -------------------------
# 1) Patient-level label for stratification
# IMPORTANT: use df_work, not df
# -------------------------
patient_labels = (
    df_work.groupby("patient_id")[LABEL_COL]
           .max()
           .reset_index()
           .rename(columns={LABEL_COL: "patient_sepsis"})
)

train_pid_df, test_pid_df = train_test_split(
    patient_labels,
    test_size=0.2,
    random_state=42,
    stratify=patient_labels["patient_sepsis"]
)

train_pids = set(train_pid_df["patient_id"])
test_pids  = set(test_pid_df["patient_id"])

# Leakage check
assert train_pids.isdisjoint(test_pids), "Data leakage: overlapping patient_ids in train/test!"

# -------------------------
# 2) Masks on raw hourly df_work (for GRU/LSTM later)
# -------------------------
tr = df_work["patient_id"].isin(train_pids)
te = df_work["patient_id"].isin(test_pids)

# -------------------------
# 3) Masks on tab_df (for LR/RF fair version)
# -------------------------
tr_tab = tab_df["patient_id"].isin(train_pids)
te_tab = tab_df["patient_id"].isin(test_pids)

# -------------------------
# 4) LR/RF window-level inputs
# -------------------------
X_train_tab_df = tab_df.loc[tr_tab, TABULAR_WINDOW_FEATS].copy()
X_test_tab_df  = tab_df.loc[te_tab, TABULAR_WINDOW_FEATS].copy()

y_tr_tab = tab_df.loc[tr_tab, LABEL_COL].to_numpy()
y_te_tab = tab_df.loc[te_tab, LABEL_COL].to_numpy()

# Standardize using training set only
scaler_tab = StandardScaler()
X_tr_tab = scaler_tab.fit_transform(X_train_tab_df)
X_te_tab = scaler_tab.transform(X_test_tab_df)

# -------------------------
# 5) Summary
# -------------------------
print(f"Train window rows      : {X_tr_tab.shape[0]:,}  (patients: {len(train_pids):,})")
print(f"Test window rows       : {X_te_tab.shape[0]:,}  (patients: {len(test_pids):,})")
print(f"Train patient pos rate : {train_pid_df['patient_sepsis'].mean()*100:.2f}%")
print(f"Test patient pos rate  : {test_pid_df['patient_sepsis'].mean()*100:.2f}%")
print(f"Train window pos rate  : {y_tr_tab.mean()*100:.2f}%")
print(f"Test window pos rate   : {y_te_tab.mean()*100:.2f}%")
print(f"Feature count          : {X_tr_tab.shape[1]}")

## Step 7 – Logistic Regression & Random Forest

In [ ]:
# =========================
# Step 7A – Logistic Regression
# (fair window-level version, faster + ETA)
# =========================

from sklearn.model_selection import GroupKFold, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import loguniform
from datetime import datetime
import numpy as np
import time

def log_msg(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

# -------------------------
# 1) Prepare data
# -------------------------
log_msg("Preparing window-level data for Logistic Regression tuning...")

X_tr_lr = X_tr_tab.astype(np.float32)
X_te_lr = X_te_tab.astype(np.float32)
y_tr_lr = y_tr_tab
y_te_lr = y_te_tab

# groups must align with training rows only
train_groups_tab = tab_df.loc[tr_tab, "patient_id"].to_numpy()

log_msg(f"X_tr_lr shape: {X_tr_lr.shape}")
log_msg(f"X_te_lr shape: {X_te_lr.shape}")
log_msg(f"Train positive rate: {y_tr_lr.mean()*100:.2f}%")
log_msg(f"Test positive rate: {y_te_lr.mean()*100:.2f}%")

# -------------------------
# 2) Faster settings
# -------------------------
N_SPLITS = 2
N_ITER = 3

gkf = GroupKFold(n_splits=N_SPLITS)

lr_base = LogisticRegression(
    max_iter=800,
    class_weight="balanced",
    solver="liblinear",
    random_state=42
)

# smaller search space for speed
lr_param_dist = {
    "C": loguniform(1e-2, 10),
    "penalty": ["l2"]
}

total_fits = N_SPLITS * N_ITER
log_msg(f"Search plan: {N_SPLITS} folds × {N_ITER} candidates = {total_fits} total fits")

# rough ETA before run
rough_seconds_per_fit = 8
rough_total_minutes = (rough_seconds_per_fit * total_fits) / 60
log_msg(f"Rough ETA before start: ~{rough_total_minutes:.1f} min")

# -------------------------
# 3) Randomized search
# -------------------------
lr_search = RandomizedSearchCV(
    estimator=lr_base,
    param_distributions=lr_param_dist,
    n_iter=N_ITER,
    scoring="average_precision",
    refit=True,
    cv=gkf,
    verbose=3,
    n_jobs=1,
    pre_dispatch=1,
    random_state=42,
    return_train_score=False
)

log_msg("Starting Logistic Regression tuning...")
start_time = time.time()

lr_search.fit(X_tr_lr, y_tr_lr, groups=train_groups_tab)

elapsed = time.time() - start_time
log_msg(f"Tuning finished in {elapsed/60:.2f} minutes")
log_msg(f"Actual average time per fit: {elapsed/total_fits:.2f} sec")

# -------------------------
# 4) Final evaluation on test set
# -------------------------
lr_m = lr_search.best_estimator_
lr_prob_tab = lr_m.predict_proba(X_te_lr)[:, 1]

lr_auroc_tab = roc_auc_score(y_te_lr, lr_prob_tab)
lr_auprc_tab = average_precision_score(y_te_lr, lr_prob_tab)

log_msg(f"Best LR params: {lr_search.best_params_}")
log_msg(f"LR Test AUROC: {lr_auroc_tab:.4f}")
log_msg(f"LR Test AUPRC: {lr_auprc_tab:.4f}")

In [ ]:
# =========================
# Step 7B – Random Forest
# (fair window-level version, faster + ETA)
# =========================

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, GroupKFold
from sklearn.metrics import roc_auc_score, average_precision_score
from scipy.stats import randint
from datetime import datetime
import numpy as np
import time

def log_msg(msg):
    print(f"[{datetime.now().strftime('%H:%M:%S')}] {msg}", flush=True)

# -------------------------
# 1) Prepare data
# -------------------------
log_msg("Preparing window-level data for Random Forest tuning...")

# RF does not strictly need scaling, but we keep the same matrices for consistency
X_tr_rf = X_tr_tab.astype(np.float32)
X_te_rf = X_te_tab.astype(np.float32)
y_tr_rf = y_tr_tab
y_te_rf = y_te_tab

# groups must align with training rows only
train_groups_rf = tab_df.loc[tr_tab, "patient_id"].to_numpy()

log_msg(f"X_tr_rf shape: {X_tr_rf.shape}")
log_msg(f"X_te_rf shape: {X_te_rf.shape}")
log_msg(f"Train positive rate: {y_tr_rf.mean()*100:.2f}%")
log_msg(f"Test positive rate: {y_te_rf.mean()*100:.2f}%")

# -------------------------
# 2) Faster settings
# -------------------------
N_SPLITS = 2
N_ITER = 3

gkf = GroupKFold(n_splits=N_SPLITS)
log_msg(f"GroupKFold with {N_SPLITS} splits created.")

rf_base = RandomForestClassifier(
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

# smaller search space for speed
rf_param_dist = {
    "n_estimators": randint(80, 180),
    "max_depth": [8, 10, None],
    "min_samples_split": randint(2, 6),
    "min_samples_leaf": randint(1, 4),
    "max_features": ["sqrt"]
}

total_fits = N_SPLITS * N_ITER
log_msg(f"Search plan: {N_SPLITS} folds × {N_ITER} candidates = {total_fits} total fits")

# rough ETA before run
rough_seconds_per_fit = 20
rough_total_minutes = (rough_seconds_per_fit * total_fits) / 60
log_msg(f"Rough ETA before start: ~{rough_total_minutes:.1f} min")

# -------------------------
# 3) Randomized search
# -------------------------
rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=rf_param_dist,
    n_iter=N_ITER,
    scoring="average_precision",
    refit=True,
    cv=gkf,
    verbose=3,
    n_jobs=1,          # keep stable on Windows
    pre_dispatch=1,
    random_state=42,
    return_train_score=False
)

log_msg("Starting Random Forest tuning...")
start_time = time.time()

rf_search.fit(X_tr_rf, y_tr_rf, groups=train_groups_rf)

elapsed = time.time() - start_time
log_msg(f"Tuning finished in {elapsed/60:.2f} minutes")
log_msg(f"Actual average time per fit: {elapsed/total_fits:.2f} sec")

# -------------------------
# 4) Final evaluation on test set
# -------------------------
rf_m = rf_search.best_estimator_
log_msg("Best estimator retrieved. Generating test-set probabilities...")

rf_prob_tab = rf_m.predict_proba(X_te_rf)[:, 1]

rf_auroc_tab = roc_auc_score(y_te_rf, rf_prob_tab)
rf_auprc_tab = average_precision_score(y_te_rf, rf_prob_tab)

log_msg(f"Best RF params: {rf_search.best_params_}")
log_msg(f"RF Test AUROC: {rf_auroc_tab:.4f}")
log_msg(f"RF Test AUPRC: {rf_auprc_tab:.4f}")

In [ ]:
# =========================
# Step 7C – Summary
# =========================

result_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest"],
    "AUROC": [lr_auroc_tab, rf_auroc_tab],
    "AUPRC": [lr_auprc_tab, rf_auprc_tab]
})

display(result_df.sort_values("AUPRC", ascending=False))

## Step 8 – NumPy GRU & LSTM

In [ ]:
# =========================
# Step 8 – GRU & LSTM (12h-only, faster version)
# =========================

import numpy as np
import pandas as pd
import time
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score

# -------------------------
# 1) Reproducibility / device
# -------------------------
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------------------
# 2) Sequence dataset helpers
# -------------------------
def build_sequence_arrays(df_part, feat_cols, label_col, seq_len, scaler):
    X_list, y_list, pid_list = [], [], []

    df_part = df_part.sort_values(["patient_id", "ICULOS"]).copy()

    for pid, grp in df_part.groupby("patient_id"):
        vals = scaler.transform(grp[feat_cols].values).astype(np.float32)
        labs = grp[label_col].values.astype(np.float32)

        if len(vals) < seq_len:
            continue

        for i in range(len(vals) - seq_len + 1):
            X_list.append(vals[i:i+seq_len])
            y_list.append(labs[i + seq_len - 1])   # predict label at last hour of window
            pid_list.append(pid)

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=np.float32)
    pid_arr = np.array(pid_list)

    return X, y, pid_arr


class SeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# -------------------------
# 3) Model definition
# -------------------------
class RNNClassifier(nn.Module):
    def __init__(self, input_dim, hidden_dim, rnn_type="GRU", num_layers=1, dropout=0.2):
        super().__init__()

        rnn_dropout = dropout if num_layers > 1 else 0.0

        if rnn_type == "GRU":
            self.rnn = nn.GRU(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout
            )
        elif rnn_type == "LSTM":
            self.rnn = nn.LSTM(
                input_size=input_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=rnn_dropout
            )
        else:
            raise ValueError("rnn_type must be 'GRU' or 'LSTM'")

        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        out, _ = self.rnn(x)
        last_h = out[:, -1, :]
        last_h = self.dropout(last_h)
        logits = self.fc(last_h).squeeze(1)
        return logits


# -------------------------
# 4) Training / evaluation helpers
# -------------------------
def train_one_model(
    model,
    train_loader,
    val_loader,
    lr,
    pos_weight,
    max_epochs=8,
    patience=2,
    weight_decay=1e-4
):
    criterion = nn.BCEWithLogitsLoss(
        pos_weight=torch.tensor([pos_weight], dtype=torch.float32, device=device)
    )
    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    best_state = None
    best_val_auprc = -np.inf
    best_epoch = 0
    patience_counter = 0

    for epoch in range(max_epochs):
        epoch_start = time.time()

        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        model.eval()
        val_probs, val_true = [], []

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                logits = model(xb)
                probs = torch.sigmoid(logits).cpu().numpy()

                val_probs.append(probs)
                val_true.append(yb.numpy())

        val_probs = np.concatenate(val_probs)
        val_true = np.concatenate(val_true)

        val_auroc = roc_auc_score(val_true, val_probs)
        val_auprc = average_precision_score(val_true, val_probs)

        epoch_elapsed = time.time() - epoch_start
        print(
            f"Epoch {epoch+1}/{max_epochs} | "
            f"train_loss={np.mean(train_losses):.4f} | "
            f"val_AUROC={val_auroc:.4f} | "
            f"val_AUPRC={val_auprc:.4f} | "
            f"time={epoch_elapsed/60:.2f} min"
        )

        if val_auprc > best_val_auprc:
            best_val_auprc = val_auprc
            best_epoch = epoch + 1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print("Early stopping triggered.")
                break

    model.load_state_dict(best_state)
    return model, best_val_auprc, best_epoch


def predict_proba(model, loader):
    model.eval()
    probs = []

    with torch.no_grad():
        for xb, _ in loader:
            xb = xb.to(device)
            logits = model(xb)
            probs.append(torch.sigmoid(logits).cpu().numpy())

    return np.concatenate(probs)


# -------------------------
# 5) Build inner train/val split for tuning
# IMPORTANT: use df_work + tr
# -------------------------
train_patient_labels = (
    df_work.loc[tr].groupby("patient_id")[LABEL_COL].max().reset_index()
)

subtrain_pids, val_pids = train_test_split(
    train_patient_labels["patient_id"].values,
    test_size=0.2,
    random_state=SEED,
    stratify=train_patient_labels[LABEL_COL].values
)

subtr_mask = df_work["patient_id"].isin(subtrain_pids)
val_mask = df_work["patient_id"].isin(val_pids)

scaler_dl = StandardScaler()
scaler_dl.fit(df_work.loc[subtr_mask, ALL_FEATURES].values)

# -------------------------
# 6) Hyperparameter search space
# 12h only, lighter search
# -------------------------
SEQ_LEN_FIXED = 12
HID_CANDIDATES = [32]
LR_CANDIDATES = [5e-4, 1e-4]
DROPOUT_CANDIDATES = [0.2, 0.3]
WEIGHT_DECAY_CANDIDATES = [1e-4]

BATCH_SIZE = 256
MAX_EPOCHS = 8
PATIENCE = 2

# -------------------------
# 7) Hyperparameter tuning for BOTH models
# -------------------------
results = []

for rnn_type in ["GRU", "LSTM"]:
    print(f"\n===== Tuning {rnn_type} (12h only) =====")

    seq_len = SEQ_LEN_FIXED
    print(f"\n--- seq_len={seq_len} ---")

    t0 = time.time()
    X_subtr, y_subtr, _ = build_sequence_arrays(
        df_work.loc[subtr_mask], ALL_FEATURES, LABEL_COL, seq_len, scaler_dl
    )
    X_val, y_val, _ = build_sequence_arrays(
        df_work.loc[val_mask], ALL_FEATURES, LABEL_COL, seq_len, scaler_dl
    )
    print(f"Built sequence arrays in {(time.time()-t0)/60:.2f} min")
    print(f"Train seq shape: {X_subtr.shape} | Val seq shape: {X_val.shape}")

    train_ds = SeqDataset(X_subtr, y_subtr)
    val_ds = SeqDataset(X_val, y_val)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

    pos = y_subtr.sum()
    neg = len(y_subtr) - pos
    pos_weight = neg / max(pos, 1.0)

    total_trials = (
        len(HID_CANDIDATES) *
        len(LR_CANDIDATES) *
        len(DROPOUT_CANDIDATES) *
        len(WEIGHT_DECAY_CANDIDATES)
    )
    print(f"Total trials for {rnn_type}: {total_trials}")

    trial_no = 0
    for hidden_dim in HID_CANDIDATES:
        for lr in LR_CANDIDATES:
            for dropout in DROPOUT_CANDIDATES:
                for weight_decay in WEIGHT_DECAY_CANDIDATES:
                    trial_no += 1
                    print(
                        f"\n[{rnn_type} trial {trial_no}/{total_trials}] "
                        f"seq_len={seq_len}, hidden_dim={hidden_dim}, "
                        f"lr={lr}, dropout={dropout}, weight_decay={weight_decay}"
                    )

                    model = RNNClassifier(
                        input_dim=len(ALL_FEATURES),
                        hidden_dim=hidden_dim,
                        rnn_type=rnn_type,
                        num_layers=1,
                        dropout=dropout
                    ).to(device)

                    model, best_val_auprc, best_epoch = train_one_model(
                        model=model,
                        train_loader=train_loader,
                        val_loader=val_loader,
                        lr=lr,
                        pos_weight=pos_weight,
                        max_epochs=MAX_EPOCHS,
                        patience=PATIENCE,
                        weight_decay=weight_decay
                    )

                    results.append({
                        "model": rnn_type,
                        "seq_len": seq_len,
                        "hidden_dim": hidden_dim,
                        "lr": lr,
                        "dropout": dropout,
                        "weight_decay": weight_decay,
                        "best_epoch": best_epoch,
                        "val_AUPRC": best_val_auprc
                    })

results_df = pd.DataFrame(results).sort_values(["model", "val_AUPRC"], ascending=[True, False])

print("\nTop tuning results by model:")
display(results_df.groupby("model", group_keys=False).head(5))

best_gru_cfg = (
    results_df[results_df["model"] == "GRU"]
    .sort_values("val_AUPRC", ascending=False)
    .iloc[0]
    .to_dict()
)

best_lstm_cfg = (
    results_df[results_df["model"] == "LSTM"]
    .sort_values("val_AUPRC", ascending=False)
    .iloc[0]
    .to_dict()
)

print("\nBest GRU configuration:", best_gru_cfg)
print("Best LSTM configuration:", best_lstm_cfg)

# -------------------------
# 8) Final train/val split for BOTH final models
# -------------------------
train_patient_labels_final = (
    df_work.loc[tr].groupby("patient_id")[LABEL_COL].max().reset_index()
)

final_train_pids, final_val_pids = train_test_split(
    train_patient_labels_final["patient_id"].values,
    test_size=0.15,
    random_state=SEED,
    stratify=train_patient_labels_final[LABEL_COL].values
)

final_tr_mask = df_work["patient_id"].isin(final_train_pids)
final_val_mask = df_work["patient_id"].isin(final_val_pids)

scaler_dl_final = StandardScaler()
scaler_dl_final.fit(df_work.loc[final_tr_mask, ALL_FEATURES].values)

def run_final_model(model_name, cfg):
    seq_len = int(cfg["seq_len"])
    hidden_dim = int(cfg["hidden_dim"])
    lr = float(cfg["lr"])
    dropout = float(cfg["dropout"])
    weight_decay = float(cfg["weight_decay"])

    print(f"\n===== Final training: {model_name} =====")
    print(
        f"seq_len={seq_len}, hidden_dim={hidden_dim}, "
        f"lr={lr}, dropout={dropout}, weight_decay={weight_decay}"
    )

    t0 = time.time()
    X_final_tr, y_final_tr, _ = build_sequence_arrays(
        df_work.loc[final_tr_mask], ALL_FEATURES, LABEL_COL, seq_len, scaler_dl_final
    )
    X_final_val, y_final_val, _ = build_sequence_arrays(
        df_work.loc[final_val_mask], ALL_FEATURES, LABEL_COL, seq_len, scaler_dl_final
    )
    X_te_seq, y_te_seq, _ = build_sequence_arrays(
        df_work.loc[te], ALL_FEATURES, LABEL_COL, seq_len, scaler_dl_final
    )
    print(f"Built final arrays in {(time.time()-t0)/60:.2f} min")
    print(f"Train seq shape: {X_final_tr.shape} | Val seq shape: {X_final_val.shape} | Test seq shape: {X_te_seq.shape}")

    train_ds_final = SeqDataset(X_final_tr, y_final_tr)
    val_ds_final = SeqDataset(X_final_val, y_final_val)
    test_ds_final = SeqDataset(X_te_seq, y_te_seq)

    train_loader_final = DataLoader(train_ds_final, batch_size=BATCH_SIZE, shuffle=True)
    val_loader_final = DataLoader(val_ds_final, batch_size=BATCH_SIZE, shuffle=False)
    test_loader_final = DataLoader(test_ds_final, batch_size=BATCH_SIZE, shuffle=False)

    pos = y_final_tr.sum()
    neg = len(y_final_tr) - pos
    pos_weight_final = neg / max(pos, 1.0)

    final_model = RNNClassifier(
        input_dim=len(ALL_FEATURES),
        hidden_dim=hidden_dim,
        rnn_type=model_name,
        num_layers=1,
        dropout=dropout
    ).to(device)

    final_model, final_val_auprc, final_best_epoch = train_one_model(
        model=final_model,
        train_loader=train_loader_final,
        val_loader=val_loader_final,
        lr=lr,
        pos_weight=pos_weight_final,
        max_epochs=MAX_EPOCHS,
        patience=PATIENCE,
        weight_decay=weight_decay
    )

    final_probs = predict_proba(final_model, test_loader_final)
    final_auroc = roc_auc_score(y_te_seq, final_probs)
    final_auprc = average_precision_score(y_te_seq, final_probs)

    print(f"\nFinal {model_name} best epoch: {final_best_epoch}")
    print(f"Final {model_name} validation AUPRC: {final_val_auprc:.4f}")
    print(f"Final {model_name} Test AUROC: {final_auroc:.4f}")
    print(f"Final {model_name} Test AUPRC: {final_auprc:.4f}")

    return {
        "model_obj": final_model,
        "seq_len": seq_len,
        "y_test_seq": y_te_seq,
        "prob_test_seq": final_probs,
        "val_AUPRC": final_val_auprc,
        "best_epoch": final_best_epoch,
        "test_AUROC": final_auroc,
        "test_AUPRC": final_auprc
    }

gru_final = run_final_model("GRU", best_gru_cfg)
lstm_final = run_final_model("LSTM", best_lstm_cfg)

# -------------------------
# 9) Save outputs for later steps
# -------------------------
gru_m = gru_final["model_obj"]
gru_prob_te = gru_final["prob_test_seq"]
gru_y_seq_te = gru_final["y_test_seq"]
gru_best_seq_len = gru_final["seq_len"]

lstm_m = lstm_final["model_obj"]
lstm_prob_te = lstm_final["prob_test_seq"]
lstm_y_seq_te = lstm_final["y_test_seq"]
lstm_best_seq_len = lstm_final["seq_len"]

# for backward compatibility if later code expects y_seq_te
y_seq_te = gru_y_seq_te

seq_model_summary_df = pd.DataFrame([
    {
        "Model": "GRU",
        "Best_seq_len": gru_best_seq_len,
        "Val_AUPRC": gru_final["val_AUPRC"],
        "Best_epoch": gru_final["best_epoch"],
        "Test_AUROC": gru_final["test_AUROC"],
        "Test_AUPRC": gru_final["test_AUPRC"]
    },
    {
        "Model": "LSTM",
        "Best_seq_len": lstm_best_seq_len,
        "Val_AUPRC": lstm_final["val_AUPRC"],
        "Best_epoch": lstm_final["best_epoch"],
        "Test_AUROC": lstm_final["test_AUROC"],
        "Test_AUPRC": lstm_final["test_AUPRC"]
    }
])

print("\nFinal sequence model summary:")
display(seq_model_summary_df.sort_values("Test_AUPRC", ascending=False))

## Step 9 – Model Evaluation

In [ ]:
# =========================
# Step 9 – Model Evaluation (fair-version compatible)
# =========================

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)
import numpy as np
import pandas as pd
import os

def evaluate(name, yt, yp, threshold=0.5):
    yt = np.asarray(yt).reshape(-1)
    yp = np.asarray(yp).reshape(-1)

    # avoid numerical issues
    yp = np.clip(yp, 1e-9, 1 - 1e-9)
    yd = (yp >= threshold).astype(int)

    auc = roc_auc_score(yt, yp)
    ap = average_precision_score(yt, yp)

    tn, fp, fn, tp = confusion_matrix(yt, yd, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn + 1e-9)
    specificity = tn / (tn + fp + 1e-9)
    precision = tp / (tp + fp + 1e-9)
    npv = tn / (tn + fn + 1e-9)
    accuracy = (tp + tn) / (tp + tn + fp + fn + 1e-9)

    return {
        "Model": name,
        "AUROC": auc,
        "AUPRC": ap,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "Precision": precision,
        "NPV": npv,
        "TP": int(tp),
        "FP": int(fp),
        "TN": int(tn),
        "FN": int(fn)
    }

# fair-version compatible variables
metrics = [
    evaluate("Logistic Regression", y_te_tab, lr_prob_tab),
    evaluate("Random Forest", y_te_tab, rf_prob_tab),
    evaluate("GRU", gru_y_seq_te, gru_prob_te),
    evaluate("LSTM", lstm_y_seq_te, lstm_prob_te)
]

metrics_df = pd.DataFrame(metrics).sort_values("AUPRC", ascending=False)

# save csv
metrics_path = os.path.join(OUTPUT_DIR, "model_performance_fair.csv")
metrics_df.to_csv(metrics_path, index=False)

# styled display
display(
    metrics_df.style
    .format({
        "AUROC": "{:.4f}",
        "AUPRC": "{:.4f}",
        "Accuracy": "{:.4f}",
        "Sensitivity": "{:.4f}",
        "Specificity": "{:.4f}",
        "Precision": "{:.4f}",
        "NPV": "{:.4f}"
    })
    .background_gradient(subset=["AUROC", "AUPRC"], cmap="Greens")
    .hide(axis="index")
    .set_caption("Model Performance on Test Set (fair-version compatible)")
)

print(f"Saved -> {metrics_path}")

## Step 10 – Clinical Early Warning Metrics

In [ ]:
# =========================
# Step 10 – Clinical Early Warning Metrics (STRICT, fair-version compatible)
# =========================

from collections import defaultdict
import numpy as np
import pandas as pd
import os

# -------------------------
# helper 1: find all sustained-alert starts
# -------------------------
def find_sustained_alert_indices(alert_flags, k=3):
    starts = []
    count = 0

    for i, flag in enumerate(alert_flags):
        if flag:
            count += 1
            if count >= k:
                start = i - k + 1
                if len(starts) == 0 or starts[-1] != start:
                    starts.append(start)
        else:
            count = 0

    return starts


# -------------------------
# helper 2: count sustained false-alarm episodes
# -------------------------
def count_sustained_alert_episodes(flags, k=3):
    fa_count = 0
    i = 0

    while i <= len(flags) - k:
        if all(flags[i:i+k]):
            fa_count += 1
            while i < len(flags) and flags[i]:
                i += 1
        else:
            i += 1

    return fa_count


# -------------------------
# strict metric for flat models (LR / RF)
# df_part must already align row-by-row with prob_arr
# -------------------------
def lead_fa_flat_sustained_strict(
    df_part,
    pid_list,
    prob_arr,
    label_col,
    threshold=0.6,
    k=3,
    max_lead_hours=24,
    use_last_alert=True
):
    df_p = (
        df_part[df_part["patient_id"].isin(pid_list)]
        .sort_values(["patient_id", "ICULOS"])
        .reset_index(drop=True)
        .copy()
    )
    df_p["prob"] = prob_arr

    leads = []
    fas = []

    for pid, grp in df_p.groupby("patient_id"):
        grp = grp.reset_index(drop=True)

        pos_idx = grp.index[grp[label_col] == 1]
        fp = pos_idx.min() if len(pos_idx) > 0 else np.nan

        # false alarms before onset
        pre = grp if pd.isna(fp) else grp.iloc[:int(fp)]
        pre_flags = (pre["prob"] >= threshold).tolist()
        fa_count = count_sustained_alert_episodes(pre_flags, k=k)
        fas.append(fa_count)

        # lead time
        if not pd.isna(fp):
            full_flags = (grp["prob"] >= threshold).tolist()

            valid_flags = [
                flag if (idx < int(fp) and idx >= max(0, int(fp) - max_lead_hours)) else False
                for idx, flag in enumerate(full_flags)
            ]

            starts = find_sustained_alert_indices(valid_flags, k=k)

            if len(starts) > 0:
                chosen_start = starts[-1] if use_last_alert else starts[0]
                leads.append(int(fp - chosen_start))

    return np.mean(leads) if len(leads) > 0 else 0.0, np.mean(fas) if len(fas) > 0 else 0.0


# -------------------------
# strict metric for sequence models (GRU / LSTM)
# -------------------------
def lead_fa_seq_sustained_strict(
    df_part,
    pid_list,
    seq_probs,
    seq_len,
    label_col,
    threshold=0.6,
    k=3,
    max_lead_hours=24,
    use_last_alert=True
):
    df_p = (
        df_part[df_part["patient_id"].isin(pid_list)]
        .sort_values(["patient_id", "ICULOS"])
        .copy()
    )

    prob_map = defaultdict(dict)
    idx = 0

    for pid, grp in df_p.groupby("patient_id"):
        grp = grp.sort_values("ICULOS").reset_index(drop=True)
        n_seq = max(0, len(grp) - seq_len + 1)

        for i in range(n_seq):
            pred_hour_idx = i + seq_len - 1
            if idx < len(seq_probs):
                prob_map[pid][pred_hour_idx] = float(seq_probs[idx])
            idx += 1

    leads = []
    fas = []

    for pid, grp in df_p.groupby("patient_id"):
        grp = grp.sort_values("ICULOS").reset_index(drop=True)
        pm = prob_map.get(pid, {})

        pos_idx = grp.index[grp[label_col] == 1]
        fp = pos_idx.min() if len(pos_idx) > 0 else np.nan

        flags = []
        for h in range(len(grp)):
            p = pm.get(h, 0.0)
            flags.append(p >= threshold)

        pre_flags = flags if pd.isna(fp) else flags[:int(fp)]
        fa_count = count_sustained_alert_episodes(pre_flags, k=k)
        fas.append(fa_count)

        if not pd.isna(fp):
            valid_flags = [
                flag if (idx_h < int(fp) and idx_h >= max(0, int(fp) - max_lead_hours)) else False
                for idx_h, flag in enumerate(flags)
            ]

            starts = find_sustained_alert_indices(valid_flags, k=k)

            if len(starts) > 0:
                chosen_start = starts[-1] if use_last_alert else starts[0]
                leads.append(int(fp - chosen_start))

    return np.mean(leads) if len(leads) > 0 else 0.0, np.mean(fas) if len(fas) > 0 else 0.0


# -------------------------
# Evaluate strict sustained-alert metrics
# -------------------------

# test patient IDs from df_work / current split
test_pids = set(df_work.loc[te, "patient_id"].unique())

# For LR/RF:
# use tab_df test endpoints, rename ICULOS_end -> ICULOS so the helper works unchanged
df_te_flat = (
    tab_df.loc[te_tab]
    .copy()
    .rename(columns={"ICULOS_end": "ICULOS"})
    .sort_values(["patient_id", "ICULOS"])
    .reset_index(drop=True)
)

# For GRU/LSTM:
# use the hourly test rows from df_work
df_te_seq = (
    df_work.loc[te]
    .copy()
    .sort_values(["patient_id", "ICULOS"])
)

# stricter settings
K_CONSEC = 3
THRESH = 0.6
MAX_LEAD_HOURS = 24
USE_LAST_ALERT = True

ew_rows = []

# Logistic Regression
lr_lead, lr_fa = lead_fa_flat_sustained_strict(
    df_te_flat,
    test_pids,
    lr_prob_tab,
    LABEL_COL,
    threshold=THRESH,
    k=K_CONSEC,
    max_lead_hours=MAX_LEAD_HOURS,
    use_last_alert=USE_LAST_ALERT
)
ew_rows.append({
    "Model": "Logistic Regression",
    "Alert Definition": f"{K_CONSEC} consecutive hours >= {THRESH}",
    "Lead Time Window": f"within {MAX_LEAD_HOURS}h before onset",
    "Mean Lead Time (h)": round(lr_lead, 2),
    "Mean False Alarms/Patient": round(lr_fa, 2)
})

# Random Forest
rf_lead, rf_fa = lead_fa_flat_sustained_strict(
    df_te_flat,
    test_pids,
    rf_prob_tab,
    LABEL_COL,
    threshold=THRESH,
    k=K_CONSEC,
    max_lead_hours=MAX_LEAD_HOURS,
    use_last_alert=USE_LAST_ALERT
)
ew_rows.append({
    "Model": "Random Forest",
    "Alert Definition": f"{K_CONSEC} consecutive hours >= {THRESH}",
    "Lead Time Window": f"within {MAX_LEAD_HOURS}h before onset",
    "Mean Lead Time (h)": round(rf_lead, 2),
    "Mean False Alarms/Patient": round(rf_fa, 2)
})

# GRU
gru_lead, gru_fa = lead_fa_seq_sustained_strict(
    df_te_seq,
    test_pids,
    gru_prob_te,
    gru_best_seq_len,
    LABEL_COL,
    threshold=THRESH,
    k=K_CONSEC,
    max_lead_hours=MAX_LEAD_HOURS,
    use_last_alert=USE_LAST_ALERT
)
ew_rows.append({
    "Model": "GRU",
    "Alert Definition": f"{K_CONSEC} consecutive hours >= {THRESH}",
    "Lead Time Window": f"within {MAX_LEAD_HOURS}h before onset",
    "Mean Lead Time (h)": round(gru_lead, 2),
    "Mean False Alarms/Patient": round(gru_fa, 2)
})

# LSTM
lstm_lead, lstm_fa = lead_fa_seq_sustained_strict(
    df_te_seq,
    test_pids,
    lstm_prob_te,
    lstm_best_seq_len,
    LABEL_COL,
    threshold=THRESH,
    k=K_CONSEC,
    max_lead_hours=MAX_LEAD_HOURS,
    use_last_alert=USE_LAST_ALERT
)
ew_rows.append({
    "Model": "LSTM",
    "Alert Definition": f"{K_CONSEC} consecutive hours >= {THRESH}",
    "Lead Time Window": f"within {MAX_LEAD_HOURS}h before onset",
    "Mean Lead Time (h)": round(lstm_lead, 2),
    "Mean False Alarms/Patient": round(lstm_fa, 2)
})

ew_df = pd.DataFrame(ew_rows)
ew_path = os.path.join(OUTPUT_DIR, "early_warning_metrics_strict_fair.csv")
ew_df.to_csv(ew_path, index=False)

display(
    ew_df.style
    .hide(axis="index")
    .set_caption("Clinical Early Warning Metrics (STRICT, fair-version compatible)")
)

print(f"Saved -> {ew_path}")

## Step 11 – Feature Importance (Random Forest)

In [ ]:
# =========================
# Step 11 – Feature Importance Analysis (with parsed feature labels)
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

importances = pd.Series(rf_m.feature_importances_, index=TABULAR_WINDOW_FEATS)
top20 = importances.nlargest(20).sort_values()

top20_df = top20.reset_index()
top20_df.columns = ["Feature", "Importance"]

# Split feature name into base variable + summary type
def split_feature_name(feat):
    for suffix in ["_last", "_mean12", "_std12", "_delta12", "_min12", "_max12"]:
        if feat.endswith(suffix):
            return feat.replace(suffix, ""), suffix.replace("_", "")
    return feat, "raw"

top20_df[["BaseFeature", "SummaryType"]] = top20_df["Feature"].apply(
    lambda x: pd.Series(split_feature_name(x))
)

fi_path = os.path.join(OUTPUT_DIR, "feature_importance.csv")
top20_df.to_csv(fi_path, index=False)

fig, ax = plt.subplots(figsize=(10, 7))
colors = plt.cm.Greens(np.linspace(0.3, 0.9, len(top20_df)))

ax.barh(top20_df["Feature"], top20_df["Importance"], color=colors, edgecolor="white")
ax.set_xlabel("Mean Decrease Impurity")
ax.set_title("Top-20 Feature Importances (Random Forest)")
ax.grid(axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Saved → {fi_path}")

## Step 12 – Publication Figure Dashboard

## ✅ Pipeline Complete

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np

# =========================
# 1) Model performance bar chart
# =========================
perf_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "GRU", "LSTM"],
    "AUROC": [0.7508, 0.7716, 0.7582, 0.7437],
    "AUPRC": [0.0605, 0.0669, 0.0577, 0.0558]
})

fig, ax = plt.subplots(figsize=(8, 5))
x = np.arange(len(perf_df))
w = 0.35

ax.bar(x - w/2, perf_df["AUROC"], width=w, label="AUROC")
ax.bar(x + w/2, perf_df["AUPRC"], width=w, label="AUPRC")

ax.set_xticks(x)
ax.set_xticklabels(perf_df["Model"], rotation=20, ha="right")
ax.set_ylabel("Score")
ax.set_title("Model Performance Comparison")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 2) Early warning comparison
# =========================
ew_df = pd.DataFrame({
    "Model": ["Logistic Regression", "Random Forest", "GRU", "LSTM"],
    "Mean Lead Time (h)": [6.88, 5.99, 5.85, 4.47],
    "Mean False Alarms/Patient": [0.54, 0.27, 0.36, 0.31]
})

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(
    ew_df["Mean False Alarms/Patient"],
    ew_df["Mean Lead Time (h)"],
    s=100
)

for _, row in ew_df.iterrows():
    ax.text(
        row["Mean False Alarms/Patient"] + 0.005,
        row["Mean Lead Time (h)"] + 0.03,
        row["Model"],
        fontsize=9
    )

ax.set_xlabel("Mean False Alarms per Patient")
ax.set_ylabel("Mean Lead Time (h)")
ax.set_title("Early Warning Trade-off")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 4) PR curve (best for imbalance)
# =========================
from sklearn.metrics import precision_recall_curve, auc

curves = {
    "Logistic Regression": (y_te_tab, lr_prob_tab),
    "Random Forest": (y_te_tab, rf_prob_tab),
    "GRU": (gru_y_seq_te, gru_prob_te),
    "LSTM": (lstm_y_seq_te, lstm_prob_te),
}

fig, ax = plt.subplots(figsize=(7, 5))

for model_name, (y_true, y_prob) in curves.items():
    precision, recall, _ = precision_recall_curve(y_true, y_prob)
    pr_auc = auc(recall, precision)
    ax.plot(recall, precision, label=f"{model_name} (AUPRC={pr_auc:.3f})")

baseline = np.mean(y_te_tab)
ax.axhline(baseline, linestyle="--", label=f"No-skill baseline={baseline:.3f}")

ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 5) Dual-axis bar chart for early warning
# =========================
fig, ax1 = plt.subplots(figsize=(8, 5))
x = np.arange(len(ew_df))

ax1.bar(x - 0.2, ew_df["Mean Lead Time (h)"], width=0.4, label="Lead Time (h)")
ax1.set_ylabel("Mean Lead Time (h)")
ax1.set_xticks(x)
ax1.set_xticklabels(ew_df["Model"], rotation=20, ha="right")
ax1.set_title("Early Warning Metrics by Model")
ax1.grid(axis="y", alpha=0.3)

ax2 = ax1.twinx()
ax2.bar(x + 0.2, ew_df["Mean False Alarms/Patient"], width=0.4, label="False Alarms/Patient")
ax2.set_ylabel("Mean False Alarms/Patient")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.show()